# **##  Ingest drivers.json **file****

1. Read the file using spark dataframe reader API
2. Define and enforce schema (preserve the nested structure)
3. Add Metadata Columns 
- Source File
- Ingestion Timestamp
4. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# Define source file and table name:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

# **Step 1- read the JSON file using the dataframe reader **API****

In [0]:
# Define Schema (NESTED ONE TO RESPECT THE NESTED STRUCTURE OF THE DATA!!!) :
from pyspark.sql.types import StructType, StructField, StringType, DateType

# Inner schema
name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

# OUTER schma
drivers_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType())
])

In [0]:
# Read data:

drivers_df = (
    spark.read
        .format('json')
        # .option('inferSchema','true')
        .option('mode', 'FAILFAST')
        .schema(drivers_schema)
        .load(source_file)
)

In [0]:
display(drivers_df)

# **Step** **2** - **Add** **Metadata** **Columns**

- Source File 
- Ingestion Timestamp

In [0]:
from pyspark.sql import functions as F

drivers_final_df = add_ingestion_metadata(drivers_df)

In [0]:
display(drivers_final_df)

# **Step 3 - Write to bronze delta table**

In [0]:
(
    drivers_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
%sql
SELECT * FROM formula1.bronze.drivers

In [0]:
display(spark.table(table_name))